In [30]:
import warnings

import numpy as np
import pandas as pd
import sklearn.preprocessing
from sklearn.preprocessing import StandardScaler, MinMaxScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
np.random.seed(42)

In [41]:
df_tr = pd.read_csv('../../data/comp2/train.csv')
df_ts = pd.read_csv('../../data/comp2/test.csv')
df_tr.head(5)

,id,title,body,label
0,0,Ok guys I have a dilemma for you,Ok so I’m thinking about asking out this girl ...,0.0
1,1,Update from my cringe overdramatic ass post,56 days ago from today I posted about my then-...,0.0
2,2,"Don't know if this is allowed, I work in youth...",NaN,0.0
3,3,So I think I just ascended to a newer level of...,So I was rebattling the elite four in Pokemon ...,0.0
4,4,My friend got a gf before me,But I have more reddit karma so I think I win 😎,0.0


In [59]:
def flatten(x):
    return x.ravel()

flatten_transformer = FunctionTransformer(flatten, validate=False)

trainData = df_tr.copy()
trainData = trainData.copy().dropna().drop(['id'], axis=1)
testData = df_ts.copy()

test_ids = testData['id']

y = trainData['label'].astype(int)
X = trainData.drop('label', axis = 1)
X_test = testData.drop(['id'], axis=1)

numFeatures = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
catFeatures = X.select_dtypes(exclude=['int64', 'float64']).columns.tolist()

textTransformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='')),
    ('flatten', flatten_transformer),
    ('tfidf', TfidfVectorizer(max_features=5000, max_df=0.95, ngram_range=(1, 3))),
])
charTransformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='')),
    ('flatten', flatten_transformer),
    ('tfidf', TfidfVectorizer(max_features=2000, ngram_range=(3, 5), analyzer='char')),
])

preprocessor = ColumnTransformer([
    ('text', textTransformer, ['body']),
    ('title', textTransformer, ['title']),
    ('ctitle', charTransformer, ['title'])
])

best_score = 0
best_model = None

'''params = {
    "classifier__max_depth": [2, 4, 5],
    "classifier__n_estimators": [100, 150, 200],
    "classifier__tree_method": ['approx'],
    "classifier__lambda": [0, 1],
    "classifier__alpha": [0, 1]
}'''

params = {
    "classifier__max_depth": [4, 6],
    "classifier__n_estimators": [224, 400],
    "classifier__tree_method": ['approx'],
    "classifier__lambda": [1],
    "classifier__alpha": [1],
    "classifier__num_parallel_tree": [2],
    "classifier__eta": [0.4]
}

xgbc = XGBClassifier(seed=42, gamma=0)

pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', xgbc)
])

grid = GridSearchCV(pipeline, params, cv=5, scoring='f1', n_jobs=1)
grid.fit(X, y)

mean_score = grid.best_score_

print(f"xgb: {mean_score:.6f}")
 
if mean_score > best_score:
    best_score = mean_score
    best_model = grid.best_estimator_
    print(f"best_params: {grid.best_params_}")

try:
    best_model.fit(X, y)
    final_preds = best_model.predict(X_test)

    submission = pd.DataFrame({
        'id': test_ids, 
        'label': final_preds.astype(int)
    })
    submission.to_csv('submission.csv', index=False)
except ValueError:
    pass

xgb: 0.781924
best_params: {'classifier__alpha': 1, 'classifier__eta': 0.4, 'classifier__lambda': 1, 'classifier__max_depth': 4, 'classifier__n_estimators': 224, 'classifier__num_parallel_tree': 2, 'classifier__tree_method': 'approx'}


In [62]:
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC


def build_text(df: pd.DataFrame) -> pd.Series:
    title = df["title"].fillna("")
    body = df["body"].fillna("")
    return (title + " [SEP] " + body).str.strip()


def build_models() -> list[Pipeline]:
    lr_model = Pipeline(
        [
            (
                "features",
                FeatureUnion(
                    [
                        (
                            "word_tfidf",
                            TfidfVectorizer(
                                analyzer="word",
                                ngram_range=(1, 2),
                                min_df=2,
                                max_df=0.95,
                                strip_accents="unicode",
                                lowercase=True,
                                sublinear_tf=True,
                            ),
                        ),
                        (
                            "char_tfidf",
                            TfidfVectorizer(
                                analyzer="char_wb",
                                ngram_range=(3, 5),
                                min_df=2,
                                strip_accents="unicode",
                                lowercase=True,
                                sublinear_tf=True,
                            ),
                        ),
                    ]
                ),
            ),
            (
                "clf",
                LogisticRegression(
                    C=4.0,
                    max_iter=2500,
                    solver="liblinear",
                    class_weight="balanced",
                ),
            ),
        ]
    )

    svc_base = Pipeline(
        [
            (
                "tfidf",
                TfidfVectorizer(
                    ngram_range=(1, 3),
                    min_df=3,
                    max_df=0.96,
                    strip_accents="unicode",
                    lowercase=True,
                    sublinear_tf=True,
                ),
            ),
            ("clf", LinearSVC(C=1.2)),
        ]
    )
    svc_model = CalibratedClassifierCV(svc_base, method="sigmoid", cv=3)
    return [lr_model, svc_model]


def find_best_threshold(y_true: np.ndarray, probabilities: np.ndarray) -> tuple[float, float]:
    best_threshold = 0.5
    best_f1 = -1.0
    for threshold in np.arange(0.3, 0.74, 0.01):
        pred = (probabilities >= threshold).astype(int)
        score = f1_score(y_true, pred)
        if score > best_f1:
            best_f1 = score
            best_threshold = float(threshold)
    return best_threshold, best_f1


def oof_ensemble_predictions(
    models: list[Pipeline],
    x_train: pd.Series,
    y_train: pd.Series,
    x_eval: pd.Series,
    n_splits: int = 5,
) -> tuple[np.ndarray, np.ndarray]:
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof = np.zeros(len(x_train), dtype=float)
    eval_pred = np.zeros(len(x_eval), dtype=float)

    for model in models:
        model_oof = np.zeros(len(x_train), dtype=float)
        model_eval = np.zeros((n_splits, len(x_eval)), dtype=float)

        for fold_idx, (fit_idx, val_idx) in enumerate(cv.split(x_train, y_train)):
            x_fit, y_fit = x_train.iloc[fit_idx], y_train.iloc[fit_idx]
            x_val_fold = x_train.iloc[val_idx]
            fitted = clone(model)
            fitted.fit(x_fit, y_fit)
            model_oof[val_idx] = fitted.predict_proba(x_val_fold)[:, 1]
            model_eval[fold_idx] = fitted.predict_proba(x_eval)[:, 1]

        oof += model_oof / len(models)
        eval_pred +=model_eval.mean(axis=0) / len(models)

    return oof, eval_pred

train_df = pd.read_csv('../../data/comp2/train.csv')
test_df = pd.read_csv('../../data/comp2/test.csv')

X = build_text(train_df)
y = train_df["label"].astype(int)
X_test = build_text(test_df)

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

models = build_models()
oof_val_proba, test_proba = oof_ensemble_predictions(models, X_train, y_train, X_test)
best_threshold, best_cv_f1 = find_best_threshold(y_train.to_numpy(), oof_val_proba)
print(f"OOF F1 (best threshold={best_threshold:.2f}): {best_cv_f1:.4f}")

val_proba = np.zeros(len(X_val), dtype=float)
for model in models:
    fitted = clone(model)
    fitted.fit(X_train, y_train)
    val_proba += fitted.predict_proba(X_val)[:, 1] / len(models)

val_pred = (val_proba >= best_threshold).astype(int)
print("Validation F1:", f1_score(y_val, val_pred))
print("Validation Accuracy:", accuracy_score(y_val, val_pred))
print(classification_report(y_val, val_pred, digits=4))

test_pred = (test_proba >= best_threshold).astype(int)
submission = pd.DataFrame({"id": test_df["id"], "label": test_pred})
submission.to_csv("submissiona.csv", index=False)
print("Saved: submissiona.csv")

OOF F1 (best threshold=0.40): 0.8273
Validation F1: 0.8155111633372503
Validation Accuracy: 0.9273148148148148
              precision    recall  f1-score   support

           0     0.9611    0.9485    0.9547      1746
           1     0.7941    0.8382    0.8155       414

    accuracy                         0.9273      2160
   macro avg     0.8776    0.8933    0.8851      2160
weighted avg     0.9291    0.9273    0.9281      2160

Saved: submissiona.csv
